#### IMPORTING Libraries, Here I have used Groq and Google Gen AI to save the cost instead of using OPEN AI APIs


In [2]:
from langchain_groq import ChatGroq                          # replaces ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings  # replaces OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os
load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile")  # free on Groq

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=os.environ.get("GEMINI_API_KEY")
)

## PDF reading

In [3]:
text_data = PyPDFLoader("NovaS.pdf").load()

# Changing Metadata of the document
for page in text_data:
    page.metadata["source"] = "NovaS.pdf"

text_data

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-31T11:24:15-03:00', 'author': 'Ansh Lamba', 'moddate': '2026-03-31T11:24:15-03:00', 'source': 'NovaS.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by d

## PDF splitting making the chunks

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.split_documents(text_data)
len(chunks)

101

## Embeddings of the chunks we created

In [5]:
# embed_model = GoogleGenerativeAIEmbeddings(
#     model="gemini-embedding-001",  # stable version, no batching problems
#     google_api_key=os.environ.get("GEMINI_API_KEY")
# )
# embedded_chunks = embed_model.embed_documents([i.page_content for i in chunks])
# len(embedded_chunks)

In [6]:
from langchain_community.vectorstores import Chroma


In [7]:
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")


In [8]:
chroma_db = Chroma.from_documents(chunks, embed_model, persist_directory="./chroma_db")

## Connection retrival

In [9]:
chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)

C:\Users\Dhruv\AppData\Local\Temp\ipykernel_37788\3816121341.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db", embedding_function=embed_model)


In [10]:
chroma_db_con.similarity_search("which year was difficult for many businesses", k=3)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 18.220610978s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

## LLM and answer generation

### It's generating only the relevent content


In [ ]:
user_query = "which year was difficult for many businesses"

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)

rel_chunks_content

['The year 2020 was difficult for many businesses around the world, but NovaSphere',
 'During the first year of operations, the company worked mostly with local startups that did',
 'understanding of real-world business problems. The company also started focusing more']

### Now to the LLM and answer generation

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [11]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='More than 15 employees by the beginning of 2019. The exact number of employees in 2023 is not specified, but it is mentioned that the organization experienced steady growth during that year.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 107, 'total_tokens': 148, 'completion_time': 0.135729459, 'completion_tokens_details': None, 'prompt_time': 0.006462139, 'prompt_tokens_details': None, 'queue_time': 0.047943289, 'total_time': 0.142191598}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e1800-a58f-7660-8c23-f1d06fd5baca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 41, 'total_tokens': 148})